In [0]:
%sql
WITH active_members AS (
    SELECT CAST(cde_id AS BIGINT) AS cde_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
    WHERE snapshot_date_id = (
            SELECT MAX(snapshot_date_id)
            FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
        )
      AND plan_id IN ('annual1', 'gift-plan')
      AND renewal_status LIKE 'Active%'
),

visits AS (
    SELECT
        t.CDE_ID,
        t.VisitDateId,
        t.LocationId,
        MAX(CASE WHEN t.TicketFLAG = 'Yes' THEN 1 ELSE 0 END)     AS had_ticket,
        MAX(CASE WHEN t.ConcessionFLAG = 'Yes' THEN 1 ELSE 0 END) AS had_concession
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
    INNER JOIN active_members m ON t.CDE_ID = m.cde_id
    WHERE t.VisitDateId >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 365), 'yyyyMMdd') AS INT)
      AND t.VisitDateId <= CAST(DATE_FORMAT(CURRENT_DATE(), 'yyyyMMdd') AS INT)
    GROUP BY t.CDE_ID, t.VisitDateId, t.LocationId
),
-- Customers who had at least one concession visit in the last year
concession_customers AS (
    SELECT DISTINCT CDE_ID
    FROM visits
    WHERE had_concession >= 1
),

-- Concession item history for those customers
concession_history AS (
    SELECT
        t.CDE_ID,
        t.VisitDateId,
        t.LocationId,
        t.Transaction_Number,
        i.Item_Description,
        i.EIS_Group_Description,
        t.Quantity,
        t.Net_Revenue
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
    INNER JOIN concession_customers c ON t.CDE_ID = c.CDE_ID
    INNER JOIN cpx_dataanalytics_gold.customer360.cpx_d_item i ON t.ItemSkey = i.itemskey
    WHERE t.VisitDateId >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 365), 'yyyyMMdd') AS INT)
      AND t.VisitDateId <= CAST(DATE_FORMAT(CURRENT_DATE(), 'yyyyMMdd') AS INT)
      AND t.ConcessionFLAG = 'Yes'
),

visit_item_counts AS (
    SELECT
        CDE_ID,
        VisitDateId,
        LocationId,
        SUM(CASE
            WHEN UPPER(Item_Description) RLIKE '(POPCORN|\\bCORN\\b)'
             AND UPPER(Item_Description) NOT RLIKE 'POPCORN ?CHICKEN'
             AND UPPER(Item_Description) NOT RLIKE '(CBO|COMBO|CPN|COUPON|% OFF|STAFF|UPSELL|UPSIZE|REFILL|SEASONING|SAMPLE|CHARITY|TRAY|BUN|LID|CART|FREE|BDAY|BIRTHDAY|PROMO)'
            THEN Quantity ELSE 0 END) AS popcorn_qty,

        SUM(CASE
            WHEN UPPER(Item_Description) RLIKE '(LARGE POPCORN|LG POPCORN|POPCORN.*LARGE|POPCORN.*\\bLG\\b|LG CORN|LARGE CORN|\\bLG130\\b|170 ?OZ|130 ?OZ)'
             AND UPPER(Item_Description) NOT RLIKE 'POPCORN ?CHICKEN'
             AND UPPER(Item_Description) NOT RLIKE '(CBO|COMBO|CPN|COUPON|% OFF|STAFF|UPSELL|UPSIZE|REFILL|CHARITY|FREE|BDAY|BIRTHDAY|PROMO)'
            THEN Quantity ELSE 0 END) AS large_popcorn_qty,

        SUM(CASE
            WHEN UPPER(Item_Description) RLIKE 'HOT ?DOG'
             AND UPPER(Item_Description) NOT RLIKE '(CBO|COMBO|CPN|COUPON|STAFF|FREE|BUN|TRAY|MEAT|CHARITY|BDAY|PROMO)'
            THEN Quantity ELSE 0 END) AS hot_dog_qty,

        SUM(CASE
            WHEN UPPER(Item_Description) RLIKE '(FOUNTAIN|\\bFTN\\b|\\bFNTN\\b)'
             AND UPPER(Item_Description) NOT RLIKE '(CBO|COMBO|CPN|COUPON|% OFF|STAFF|UPSELL|UPSIZE|LID|CART|FREE|BDAY|BIRTHDAY|PROMO)'
            THEN Quantity ELSE 0 END) AS fountain_qty
    FROM concession_history
    GROUP BY CDE_ID, VisitDateId, LocationId
),

-- Final offer metrics
offer_metrics AS (
    SELECT
        COUNT(*) AS total_concession_visits,
        SUM(CASE WHEN popcorn_qty = 1  THEN 1 ELSE 0 END) AS visits_exactly_1_popcorn,
        SUM(CASE WHEN popcorn_qty >= 2 THEN 1 ELSE 0 END) AS visits_2plus_popcorn,
        SUM(CASE WHEN hot_dog_qty = 1  THEN 1 ELSE 0 END) AS visits_exactly_1_hotdog,
        SUM(CASE WHEN hot_dog_qty >= 2 THEN 1 ELSE 0 END) AS visits_2plus_hotdog,
        SUM(CASE WHEN fountain_qty >= 1 THEN 1 ELSE 0 END) AS visits_1plus_fountain,
        SUM(CASE WHEN large_popcorn_qty >= 1 AND fountain_qty = 0  THEN 1 ELSE 0 END) AS visits_lg_popcorn_no_fountain,
        SUM(CASE WHEN large_popcorn_qty >= 1 AND fountain_qty >= 1 THEN 1 ELSE 0 END) AS visits_lg_popcorn_with_fountain
    FROM visit_item_counts
)

Select * From offer_metrics

In [0]:
%sql

-- Visit-level rollup (from before): one row per CDE_ID + VisitDateId + LocationId
visits AS (
    SELECT
        t.CDE_ID,
        t.VisitDateId,
        t.LocationId,
        MAX(CASE WHEN t.TicketFLAG = 'Yes' THEN 1 ELSE 0 END)     AS had_ticket,
        MAX(CASE WHEN t.ConcessionFLAG = 'Yes' THEN 1 ELSE 0 END) AS had_concession
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
    INNER JOIN active_members m ON t.CDE_ID = m.cde_id
    WHERE t.VisitDateId >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 365), 'yyyyMMdd') AS INT)
      AND t.VisitDateId <= CAST(DATE_FORMAT(CURRENT_DATE(), 'yyyyMMdd') AS INT)
    GROUP BY t.CDE_ID, t.VisitDateId, t.LocationId
),

-- Customers with at least one concession visit (from before)
concession_customers AS (
    SELECT DISTINCT CDE_ID
    FROM visits
    WHERE had_concession = 1
),

-- Concession item history (from before)
concession_history AS (
    SELECT
        t.CDE_ID,
        t.VisitDateId,
        t.LocationId,
        i.Item_Description,
        t.Quantity
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
    INNER JOIN concession_customers c ON t.CDE_ID = c.CDE_ID
    LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_item i ON t.ItemSkey = i.itemskey
    WHERE t.VisitDateId >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 365), 'yyyyMMdd') AS INT)
      AND t.VisitDateId <= CAST(DATE_FORMAT(CURRENT_DATE(), 'yyyyMMdd') AS INT)
      AND t.ConcessionFLAG = 'Yes'
      AND t.Quantity > 0   -- exclude refunds
),

-- NEW: aggregate concession history to visit level with category counts
visit_item_counts AS (
    SELECT
        CDE_ID,
        VisitDateId,
        LocationId,
        SUM(CASE
            WHEN UPPER(Item_Description) RLIKE '(POPCORN|\\bCORN\\b)'
             AND UPPER(Item_Description) NOT RLIKE 'POPCORN ?CHICKEN'
             AND UPPER(Item_Description) NOT RLIKE '(CBO|COMBO|CPN|COUPON|% OFF|STAFF|UPSELL|UPSIZE|REFILL|SEASONING|SAMPLE|CHARITY|TRAY|BUN|LID|CART|FREE|BDAY|BIRTHDAY|PROMO)'
            THEN Quantity ELSE 0 END) AS popcorn_qty,

        SUM(CASE
            WHEN UPPER(Item_Description) RLIKE '(LARGE POPCORN|LG POPCORN|POPCORN.*LARGE|POPCORN.*\\bLG\\b|LG CORN|LARGE CORN|\\bLG130\\b|170 ?OZ|130 ?OZ)'
             AND UPPER(Item_Description) NOT RLIKE 'POPCORN ?CHICKEN'
             AND UPPER(Item_Description) NOT RLIKE '(CBO|COMBO|CPN|COUPON|% OFF|STAFF|UPSELL|UPSIZE|REFILL|CHARITY|FREE|BDAY|BIRTHDAY|PROMO)'
            THEN Quantity ELSE 0 END) AS large_popcorn_qty,

        SUM(CASE
            WHEN UPPER(Item_Description) RLIKE 'HOT ?DOG'
             AND UPPER(Item_Description) NOT RLIKE '(CBO|COMBO|CPN|COUPON|STAFF|FREE|BUN|TRAY|MEAT|CHARITY|BDAY|PROMO)'
            THEN Quantity ELSE 0 END) AS hot_dog_qty,

        SUM(CASE
            WHEN UPPER(Item_Description) RLIKE '(FOUNTAIN|\\bFTN\\b|\\bFNTN\\b)'
             AND UPPER(Item_Description) NOT RLIKE '(CBO|COMBO|CPN|COUPON|% OFF|STAFF|UPSELL|UPSIZE|LID|CART|FREE|BDAY|BIRTHDAY|PROMO)'
            THEN Quantity ELSE 0 END) AS fountain_qty
    FROM concession_history
    GROUP BY CDE_ID, VisitDateId, LocationId
),

-- Final offer metrics
offer_metrics AS (
    SELECT
        COUNT(*) AS total_concession_visits,
        SUM(CASE WHEN popcorn_qty = 1  THEN 1 ELSE 0 END) AS visits_exactly_1_popcorn,
        SUM(CASE WHEN popcorn_qty >= 2 THEN 1 ELSE 0 END) AS visits_2plus_popcorn,
        SUM(CASE WHEN hot_dog_qty = 1  THEN 1 ELSE 0 END) AS visits_exactly_1_hotdog,
        SUM(CASE WHEN hot_dog_qty >= 2 THEN 1 ELSE 0 END) AS visits_2plus_hotdog,
        SUM(CASE WHEN fountain_qty >= 1 THEN 1 ELSE 0 END) AS visits_1plus_fountain,
        SUM(CASE WHEN large_popcorn_qty >= 1 AND fountain_qty = 0  THEN 1 ELSE 0 END) AS visits_lg_popcorn_no_fountain,
        SUM(CASE WHEN large_popcorn_qty >= 1 AND fountain_qty >= 1 THEN 1 ELSE 0 END) AS visits_lg_popcorn_with_fountain
    FROM visit_item_counts
)

-- SELECT
--     total_concession_visits,
--     ROUND(100.0 * visits_exactly_1_popcorn        / total_concession_visits, 2) AS pct_exactly_1_popcorn,
--     ROUND(100.0 * visits_2plus_popcorn            / total_concession_visits, 2) AS pct_2plus_popcorn,
--     ROUND(100.0 * visits_exactly_1_hotdog         / total_concession_visits, 2) AS pct_exactly_1_hotdog,
--     ROUND(100.0 * visits_2plus_hotdog             / total_concession_visits, 2) AS pct_2plus_hotdog,
--     ROUND(100.0 * visits_1plus_fountain           / total_concession_visits, 2) AS pct_1plus_fountain,
--     ROUND(100.0 * visits_lg_popcorn_no_fountain   / total_concession_visits, 2) AS pct_lg_popcorn_no_fountain,
--     ROUND(100.0 * visits_lg_popcorn_with_fountain / total_concession_visits, 2) AS pct_lg_popcorn_with_fountain
-- FROM offer_metrics
-- ;

Select * From visit_item_counts

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# 1. Active subscribers on the latest snapshot
active_members = spark.sql("""
    SELECT CAST(cde_id AS BIGINT) AS cde_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
    WHERE snapshot_date_id = (
            SELECT MAX(snapshot_date_id)
            FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
        )
      AND plan_id IN ('annual1', 'gift-plan')
      AND renewal_status LIKE 'Active%'
""")
active_members.createOrReplaceTempView("active_members")
# active_members.cache()
# active_members.count()

In [0]:
# 2. Distinct visits in the last year for active members
visits = spark.sql("""
    SELECT
        t.CDE_ID,
        t.VisitDateId,
        t.LocationId,
        MAX(CASE WHEN t.TicketFLAG = 'Yes' THEN 1 ELSE 0 END)     AS had_ticket,
        MAX(CASE WHEN t.ConcessionFLAG = 'Yes' THEN 1 ELSE 0 END) AS had_concession
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
    INNER JOIN active_members m ON t.CDE_ID = m.cde_id
    WHERE t.VisitDateId >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 365), 'yyyyMMdd') AS INT)
      AND t.VisitDateId <= CAST(DATE_FORMAT(CURRENT_DATE(), 'yyyyMMdd') AS INT)
    GROUP BY t.CDE_ID, t.VisitDateId, t.LocationId
""")
visits.createOrReplaceTempView("visits")
# visits.cache()
# visits.count()

In [0]:
# 3. Customers with at least one concession visit
concession_customers = spark.sql("""
    SELECT DISTINCT CDE_ID
    FROM visits
    WHERE had_concession >= 1
""")
concession_customers.createOrReplaceTempView("concession_customers")
# concession_customers.cache()
# concession_customers.count()

In [0]:
# 4. Concession item history for those customers
concession_history = spark.sql("""
    SELECT
        t.CDE_ID,
        t.VisitDateId,
        t.LocationId,
        t.Transaction_Number,
        i.Item_Description,
        i.EIS_Group_Description,
        t.Quantity,
        t.Net_Revenue
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
    INNER JOIN concession_customers c ON t.CDE_ID = c.CDE_ID
    INNER JOIN cpx_dataanalytics_gold.customer360.cpx_d_item i
        ON t.ItemSkey = i.itemskey
    WHERE t.VisitDateId >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 365), 'yyyyMMdd') AS INT)
      AND t.VisitDateId <= CAST(DATE_FORMAT(CURRENT_DATE(), 'yyyyMMdd') AS INT)
      AND t.ConcessionFLAG = 'Yes'
""")
concession_history.createOrReplaceTempView("concession_history")
# concession_history.cache()
# concession_history.count()



In [0]:
# 5. Total distinct visits per customer
#    `visits` is already at the (CDE_ID, VisitDateId, LocationId) grain,
#    so COUNT(*) per CDE_ID = distinct visits (independent of ticket/concession flags)
customer_visit_totals = spark.sql("""
    SELECT
        CDE_ID,
        COUNT(*) AS total_visits
    FROM visits
    GROUP BY CDE_ID
""")
customer_visit_totals.createOrReplaceTempView("customer_visit_totals")
# customer_visit_totals.cache()
# customer_visit_totals.count()

In [0]:
# 6. Per-customer concession summary (one row per CDE_ID)
visit_item_counts = spark.sql(r"""
    WITH per_visit AS (
        SELECT
            CDE_ID,
            VisitDateId,
            LocationId,

            SUM(CASE
                WHEN UPPER(Item_Description) RLIKE '(POPCORN|\\bCORN\\b)'
                 AND UPPER(Item_Description) NOT RLIKE 'POPCORN ?CHICKEN'
                 AND UPPER(Item_Description) NOT RLIKE '(CBO|COMBO|CPN|COUPON|% OFF|STAFF|UPSELL|UPSIZE|REFILL|SEASONING|SAMPLE|CHARITY|TRAY|BUN|LID|CART|FREE|BDAY|BIRTHDAY|PROMO)'
                THEN Quantity ELSE 0 END) AS popcorn_qty,

            SUM(CASE
                WHEN UPPER(Item_Description) RLIKE '(LARGE POPCORN|LG POPCORN|POPCORN.*LARGE|POPCORN.*\\bLG\\b|LG CORN|LARGE CORN|\\bLG130\\b|170 ?OZ|130 ?OZ)'
                 AND UPPER(Item_Description) NOT RLIKE 'POPCORN ?CHICKEN'
                 AND UPPER(Item_Description) NOT RLIKE '(CBO|COMBO|CPN|COUPON|% OFF|STAFF|UPSELL|UPSIZE|REFILL|CHARITY|FREE|BDAY|BIRTHDAY|PROMO)'
                THEN Quantity ELSE 0 END) AS large_popcorn_qty,

            SUM(CASE
                WHEN UPPER(Item_Description) RLIKE 'HOT ?DOG'
                 AND UPPER(Item_Description) NOT RLIKE '(CBO|COMBO|CPN|COUPON|STAFF|FREE|BUN|TRAY|MEAT|CHARITY|BDAY|PROMO)'
                THEN Quantity ELSE 0 END) AS hot_dog_qty,

            SUM(CASE
                WHEN UPPER(Item_Description) RLIKE '(FOUNTAIN|\\bFTN\\b|\\bFNTN\\b)'
                 AND UPPER(Item_Description) NOT RLIKE '(CBO|COMBO|CPN|COUPON|% OFF|STAFF|UPSELL|UPSIZE|LID|CART|FREE|BDAY|BIRTHDAY|PROMO)'
                THEN Quantity ELSE 0 END) AS fountain_qty
        FROM concession_history
        GROUP BY CDE_ID, VisitDateId, LocationId
    )
    SELECT
        p.CDE_ID,
        v.total_visits,

        COUNT(*) AS total_concession_visits,

        SUM(CASE WHEN p.popcorn_qty = 1  THEN 1 ELSE 0 END) AS visits_exactly_1_popcorn,
        SUM(CASE WHEN p.popcorn_qty >= 2 THEN 1 ELSE 0 END) AS visits_2plus_popcorn,

        SUM(CASE WHEN p.hot_dog_qty = 1  THEN 1 ELSE 0 END) AS visits_exactly_1_hotdog,
        SUM(CASE WHEN p.hot_dog_qty >= 2 THEN 1 ELSE 0 END) AS visits_2plus_hotdog,

        SUM(CASE WHEN p.fountain_qty >= 1 THEN 1 ELSE 0 END) AS visits_1plus_fountain,

        SUM(CASE WHEN p.large_popcorn_qty >= 1 AND p.fountain_qty = 0  THEN 1 ELSE 0 END) AS visits_lg_popcorn_no_fountain,
        SUM(CASE WHEN p.large_popcorn_qty >= 1 AND p.fountain_qty >= 1 THEN 1 ELSE 0 END) AS visits_lg_popcorn_with_fountain
    FROM per_visit p
    LEFT JOIN customer_visit_totals v
        ON p.CDE_ID = v.CDE_ID
    GROUP BY p.CDE_ID, v.total_visits
""")
visit_item_counts.createOrReplaceTempView("visit_item_counts")


In [0]:
visit_item_counts.display()

In [0]:
offer_modeling = spark.sql(r"""
    SELECT
        COUNT(DISTINCT CDE_ID)                 AS members,
        SUM(total_visits)                      AS total_visits,
        SUM(total_concession_visits)           AS total_concession_visits,

        -- BOGO Large Popcorn (any-size proxy)
        SUM(visits_exactly_1_popcorn) / SUM(total_visits) AS pct_visits_1_popcorn,
        SUM(visits_2plus_popcorn)     / SUM(total_visits) AS pct_visits_2plus_popcorn,

        -- BOGO Hot Dog
        SUM(visits_exactly_1_hotdog)  / SUM(total_visits) AS pct_visits_1_hotdog,
        SUM(visits_2plus_hotdog)      / SUM(total_visits) AS pct_visits_2plus_hotdog,

        -- Fountain + 500 pts
        SUM(visits_1plus_fountain)    / SUM(total_visits) AS pct_visits_1plus_fountain,

        -- Large Popcorn + Free Fountain
        SUM(visits_lg_popcorn_no_fountain)   / SUM(total_visits) AS pct_visits_lgpop_no_ftn,
        SUM(visits_lg_popcorn_with_fountain) / SUM(total_visits) AS pct_visits_lgpop_with_ftn
    FROM visit_item_counts
""")


In [0]:
offer_modeling.display()
